# 06 — Tokenizer round-trip pre-flight

**Purpose:** Before regen + Experiment 4, verify that `dataset.marked.json` survives the Gemma tokenizer without mutation or byte-fragmentation.

**Pass bar:**
1. `decode(encode(aria_label)) == aria_label` for every record (lossless).
2. Rare codepoints `²`, `€`, `£`, `—` each tokenize to ≤2 tokens (not 3–4 byte fragments).

**If (1) fails:** the tokenizer is mutating text — fix at the source before training.  
**If (2) fails:** training signal for those chars dilutes across byte tokens — consider ASCII-canonicalizing them (e.g. `m²` → `m2`).

Records under test: #14 (French-heavy), #37 (`€/m²` combo), #50 (widest codepoint diversity). Full 50-record sweep at the end.

In [1]:
import json
import collections
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

MODEL_REPO = "unsloth/gemma-4-e4b-it"
LOCAL_MODEL_DIR = "/kaggle/working/gemma4-unsloth"
DATASET_JSON = "/kaggle/input/datasets/shahfazalmohammed/civicinsight-dataset/dataset.marked.json"

In [2]:
# Authenticate with Hugging Face
from huggingface_hub import login

# Method 1: Kaggle Secrets (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle Secrets")
except Exception:
    print("Could not find HF_TOKEN in Kaggle Secrets. Set it up or login manually.")

Logged in via Kaggle Secrets


In [3]:
# one-time download of model repo (tokenizer files only needed, but full download matches smoketest)
path = snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=LOCAL_MODEL_DIR,
    ignore_patterns=["*.md"],
)
print(f"Downloaded to: {path}")

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Downloaded to: /kaggle/working/gemma4-unsloth


In [4]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
print(f"Tokenizer class: {type(tokenizer).__name__}")
print(f"Vocab size:      {tokenizer.vocab_size}")
print(f"Model max len:   {tokenizer.model_max_length}")

You are using a model of type gemma4 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


Tokenizer class: TokenizersBackend
Vocab size:      262144
Model max len:   131072


In [5]:
with open(DATASET_JSON) as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} records")

# records under test — 1-indexed as discussed, so slice -1
TEST_IDX = [14, 37, 50]
TEST_RECORDS = [(i, dataset[i - 1]) for i in TEST_IDX]
for i, rec in TEST_RECORDS:
    print(f"  #{i}: {rec['image'].split('/')[-1]}")

Loaded 50 records
  #14: social-media-stacked-highlight-50-64-fr.png
  #37: quintile-distrib.png
  #50: tourisme-powerbi.png


## Check 1 — Lossless round-trip on test records

In [6]:
def roundtrip(text, tok):
    ids = tok.encode(text, add_special_tokens=False)
    decoded = tok.decode(ids, skip_special_tokens=False)
    return ids, decoded, decoded == text

for i, rec in TEST_RECORDS:
    original = rec["aria_label"]
    ids, decoded, ok = roundtrip(original, tokenizer)
    status = "PASS" if ok else "FAIL"
    print(f"#{i}  {status}  {len(original)} chars -> {len(ids)} tokens  ({len(ids)/len(original):.2f} tok/char)")
    if not ok:
        # find first divergence
        for j, (a, b) in enumerate(zip(original, decoded)):
            if a != b:
                print(f"     first diff at char {j}: {a!r} -> {b!r}")
                print(f"     context: ...{original[max(0,j-20):j+20]}...")
                break
        if len(original) != len(decoded):
            print(f"     length mismatch: {len(original)} vs {len(decoded)}")

#14  PASS  693 chars -> 219 tokens  (0.32 tok/char)
#37  PASS  713 chars -> 239 tokens  (0.34 tok/char)
#50  PASS  1684 chars -> 524 tokens  (0.31 tok/char)


## Check 2 — Rare codepoint token efficiency

Each rare char isolated into minimal context so we see how the tokenizer splits it.

In [7]:
RARE = {
    "²": "m²",
    "€": "€40K",
    "£": "£400 000",
    "—": "text — text",
    "é": "démographiques",
    "ê": "chères",
    "É": "Émissions",
    "ç": "Français",
    "ô": "hôtel",
    "î": "maître",
    "Î": "Île",
    "è": "problème",
    "[civicinsight-v1]": "[civicinsight-v1] This chart",
}

print(f"{'char':<20} {'probe':<35} {'#tokens':<8} {'tokens (readable)'}")
print("-" * 110)
for ch, probe in RARE.items():
    ids = tokenizer.encode(probe, add_special_tokens=False)
    readable = [tokenizer.decode([i]) for i in ids]
    print(f"{ch!r:<20} {probe!r:<35} {len(ids):<8} {readable}")

char                 probe                               #tokens  tokens (readable)
--------------------------------------------------------------------------------------------------------------
'²'                  'm²'                                2        ['m', '²']
'€'                  '€40K'                              4        ['€', '4', '0', 'K']
'£'                  '£400 000'                          8        ['£', '4', '0', '0', ' ', '0', '0', '0']
'—'                  'text — text'                       3        ['text', ' —', ' text']
'é'                  'démographiques'                    3        ['d', 'ém', 'ographiques']
'ê'                  'chères'                            2        ['ch', 'ères']
'É'                  'Émissions'                         2        ['É', 'missions']
'ç'                  'Français'                          2        ['Fr', 'ançais']
'ô'                  'hôtel'                             1        ['hôtel']
'î'                  'maîtr

## Check 3 — Sweep all 50 records

In [8]:
failures = []
tok_counts = []
for i, rec in enumerate(dataset, start=1):
    original = rec["aria_label"]
    ids, decoded, ok = roundtrip(original, tokenizer)
    tok_counts.append((i, len(original), len(ids)))
    if not ok:
        failures.append((i, original, decoded))

print(f"Total records:       {len(dataset)}")
print(f"Round-trip passes:   {len(dataset) - len(failures)}")
print(f"Round-trip failures: {len(failures)}")
print()
print(f"Token count stats (across all 50):")
token_lens = [t for _, _, t in tok_counts]
print(f"  min:    {min(token_lens)}")
print(f"  max:    {max(token_lens)}")
print(f"  mean:   {sum(token_lens)/len(token_lens):.1f}")
print(f"  median: {sorted(token_lens)[len(token_lens)//2]}")
print()
print("Longest 5 records (for max_seq_length planning):")
for i, char_len, tok_len in sorted(tok_counts, key=lambda x: -x[2])[:5]:
    print(f"  #{i:2d}  {char_len:4d} chars  {tok_len:4d} tokens")

if failures:
    print("\nFAILURES:")
    for i, orig, dec in failures:
        print(f"  #{i}")
        for j, (a, b) in enumerate(zip(orig, dec)):
            if a != b:
                print(f"    diff @ {j}: {a!r} -> {b!r} | context: ...{orig[max(0,j-15):j+15]}...")
                break

Total records:       50
Round-trip passes:   50
Round-trip failures: 0

Token count stats (across all 50):
  min:    114
  max:    524
  mean:   186.3
  median: 152

Longest 5 records (for max_seq_length planning):
  #50  1684 chars   524 tokens
  #40  1211 chars   380 tokens
  # 6  1094 chars   360 tokens
  #49  1096 chars   354 tokens
  #41   900 chars   286 tokens


## Result interpretation

- **All 50 pass round-trip + rare codepoints are ≤2 tokens** → proceed to Experiment 4 unchanged.
- **Round-trip failure on any record** → inspect the diff char; likely a lookalike (e.g. NBSP vs space, curly vs straight apostrophe). Fix in `dataset.json`, regen `dataset.marked.json`.
- **Codepoint fragments to 3+ tokens** (e.g. `²` becomes `[0xE2, 0x82, 0xB2]`) → training signal dilutes. Either accept (rare chars are a small fraction of total tokens) or ASCII-canonicalize that char in `dataset.json`.
- **Longest record approaches 2048 tokens** → bump `max_seq_length` in Experiment 4 SFTConfig.